In [ ]:
!unzip dataset.zip

Streaming output truncated to the last 5000 lines.
  inflating: training/real/recording12956.wav_norm_mono.wav  
  inflating: training/real/recording12957.wav_norm_mono.wav  
  inflating: training/real/recording12958.wav_norm_mono.wav  
  inflating: training/real/recording12959.wav_norm_mono.wav  
  inflating: training/real/recording12961.wav_norm_mono.wav  
  inflating: training/real/recording12962.wav_norm_mono.wav  
  inflating: training/real/recording12965.wav_norm_mono.wav  
  inflating: training/real/recording12966.wav_norm_mono.wav  
  inflating: training/real/recording12967.wav_norm_mono.wav  
  inflating: training/real/recording12968.wav_norm_mono.wav  
  inflating: training/real/recording12969.wav_norm_mono.wav  
  inflating: training/real/recording12970.wav_norm_mono.wav  
  inflating: training/real/recording12976.wav_norm_mono.wav  
  inflating: training/real/recording12977.wav_norm_mono.wav  
  inflating: training/real/recording12979.wav_norm_mono.wav  
  inflating: traini

In [ ]:
import os

def count_files(folder):
    return len([f for f in os.listdir(folder) if os.path.isfile(os.path.join(folder, f))])

train_real = "/content/training/real"
train_fake = "/content/training/fake"

val_real = "/content/validation/real"
val_fake = "/content/validation/fake"

test_real = "/content/testing/real"
test_fake = "/content/testing/fake"


print("TRAIN DATA")
print("Real:", count_files(train_real))
print("Fake:", count_files(train_fake))
print("Total:", count_files(train_real) + count_files(train_fake))

print("\nVALIDATION DATA")
print("Real:", count_files(val_real))
print("Fake:", count_files(val_fake))
print("Total:", count_files(val_real) + count_files(val_fake))

print("\nTEST DATA")
print("Real:", count_files(test_real))
print("Fake:", count_files(test_fake))
print("Total:", count_files(test_real) + count_files(test_fake))

TRAIN DATA
Real: 5104
Fake: 5104
Total: 10208

VALIDATION DATA
Real: 1101
Fake: 1143
Total: 2244

TEST DATA
Real: 408
Fake: 408
Total: 816


In [ ]:
import os
import librosa
import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import models
from torch.utils.data import Dataset, DataLoader
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [ ]:
def count_files(folder):
    return len([f for f in os.listdir(folder) if f.endswith(".wav")])

print("TRAIN REAL:", count_files(train_real))
print("TRAIN FAKE:", count_files(train_fake))

print("VAL REAL:", count_files(val_real))
print("VAL FAKE:", count_files(val_fake))

print("TEST REAL:", count_files(test_real))
print("TEST FAKE:", count_files(test_fake))

TRAIN REAL: 5104
TRAIN FAKE: 5104
VAL REAL: 1101
VAL FAKE: 1143
TEST REAL: 408
TEST FAKE: 408


In [ ]:
def audio_to_spec(path):

    audio, sr = librosa.load(path, sr=16000)

    max_len = 16000 * 2

    if len(audio) < max_len:
        audio = np.pad(audio, (0, max_len - len(audio)))
    else:
        audio = audio[:max_len]

    spec = librosa.feature.melspectrogram(
        y=audio,
        sr=sr,
        n_mels=128,
        n_fft=1024,
        hop_length=512
    )

    spec = librosa.power_to_db(spec, ref=np.max)

    spec = cv2.resize(spec, (224,224))

    spec = np.stack((spec,)*3, axis=0)

    spec = torch.tensor(spec).float()

    return spec


In [ ]:
class AudioDataset(Dataset):

    def __init__(self, real_folder, fake_folder):

        self.files = []
        self.labels = []

        for f in os.listdir(real_folder):
            if f.endswith(".wav"):
                self.files.append(os.path.join(real_folder,f))
                self.labels.append(0)

        for f in os.listdir(fake_folder):
            if f.endswith(".wav"):
                self.files.append(os.path.join(fake_folder,f))
                self.labels.append(1)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):

        path = self.files[idx]
        label = self.labels[idx]

        spec = audio_to_spec(path)

        return spec, torch.tensor(label)

In [ ]:
train_dataset = AudioDataset(train_real, train_fake)
val_dataset = AudioDataset(val_real, val_fake)
test_dataset = AudioDataset(test_real, test_fake)

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 10208
Val samples: 2244
Test samples: 816


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)
test_loader = DataLoader(test_dataset, batch_size=8)

In [ ]:
model = models.vgg16(pretrained=True)

model.classifier[6] = nn.Linear(4096,2)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 80.4MB/s]


In [ ]:
criterion = nn.CrossEntropyLoss()

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    patience=3,
    factor=0.5
)

In [ ]:
epochs = 20
best_val_acc = 0

for epoch in range(epochs):

    model.train()
    train_loss = 0

    for spectrograms, labels in train_loader:

        spectrograms = spectrograms.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(spectrograms)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Validation
    model.eval()

    correct = 0
    total = 0
    val_loss = 0

    with torch.no_grad():

        for spectrograms, labels in val_loader:

            spectrograms = spectrograms.to(device)
            labels = labels.to(device)

            outputs = model(spectrograms)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            predicted = torch.argmax(outputs,1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = correct / total

    print("Epoch:", epoch+1)
    print("Train Loss:", train_loss)
    print("Val Accuracy:", val_acc)
    print("--------------------------------")

    scheduler.step(val_loss)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(model.state_dict(), "best_model.pth")

        print("Best model saved")

Epoch: 1
Train Loss: 504.41160733513243
Val Accuracy: 0.8645276292335116
--------------------------------
Best model saved
Epoch: 2
Train Loss: 169.04301203131132
Val Accuracy: 0.9175579322638147
--------------------------------
Best model saved
Epoch: 3
Train Loss: 105.24165493266591
Val Accuracy: 0.9897504456327986
--------------------------------
Best model saved
Epoch: 4
Train Loss: 58.15382557714106
Val Accuracy: 0.9777183600713012
--------------------------------
Epoch: 5
Train Loss: 92.12604195407785
Val Accuracy: 0.9888591800356507
--------------------------------
Epoch: 6
Train Loss: 50.82561613173151
Val Accuracy: 0.9937611408199644
--------------------------------
Best model saved
Epoch: 7
Train Loss: 30.617583666010972
Val Accuracy: 0.9906417112299465
--------------------------------
Epoch: 8
Train Loss: 57.36835060587548
Val Accuracy: 0.9964349376114082
--------------------------------
Best model saved
Epoch: 9
Train Loss: 65.30505327596984
Val Accuracy: 0.9549910873440285

KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), "deepfake_audio_model.pth")

print("Model saved successfully!")

Model saved successfully!


In [ ]:
model.load_state_dict(torch.load("deepfake_audio_model.pth"))

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for spectrograms, labels in test_loader:

        spectrograms = spectrograms.to(device)
        labels = labels.to(device)

        outputs = model(spectrograms)

        predicted = torch.argmax(outputs,1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Test Accuracy:", correct/total)

Test Accuracy: 0.75


In [ ]:
def predict_audio(path):

    spec = audio_to_spec(path)

    spec = spec.unsqueeze(0).to(device)

    model.eval()

    with torch.no_grad():

        output = model(spec)

        pred = torch.argmax(output,1).item()

        prob = torch.softmax(output, dim=1)

        confidence = prob[0][pred].item()

    if pred == 0:
        label = "REAL AUDIO"
    else:
        label = "FAKE AUDIO"

    print("Prediction:", label)
    print("Confidence:", round(confidence*100,2), "%")

In [ ]:
predict_audio("/content/testing/fake/recording13023.wav_norm_mono.wav")

Prediction: FAKE AUDIO
Confidence: 67.45 %


In [ ]:
from google.colab import files

files.download("deepfake_audio_model.pth")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>